# 01 · Exploring the data

A first look at the Landsat-8 fire dataset: what the files are, what an image
looks like, why fire stands out in some bands, and what the fire masks contain.

Run the cells in order. If `data/raw/regions` is missing, the notebook uses the
100-patch sample in `data/raw/samples` instead.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import rasterio

DATA = Path("../data/raw")
if (DATA / "regions").exists():
    folder = DATA / "regions"
else:
    folder = DATA / "samples"
    print("Using the small sample")

# The manual annotations are a separate test set, looked at in section 9.
all_files = sorted(f for f in folder.rglob("*.tif") if "manual_annotations" not in str(f))
print(len(all_files), "files")

## 1. What files are there?

There are two kinds of files:

- **Images**: a piece of a satellite picture.
- **Masks**: for each image, a black-and-white picture where white means "fire".
  There is one mask per fire-detection algorithm (Schroeder, Murphy, Kumar-Roy).

The file name tells them apart. Split by `_`, the 8th part is the patch number
for an image (`p00367.tif`) and the algorithm's name for a mask:

- image: `LC08_L1TP_124046_20200810_20200810_01_RT_p00367.tif`
- mask: `LC08_L1TP_124046_20200810_20200810_01_RT_Schroeder_p00367.tif`

In the sample, Kumar-Roy masks are called `GOLI_v2`.

In [ ]:
def algorithm_of(path):
    """The algorithm name of a mask file, or None for an image."""
    parts = path.stem.split("_")
    if parts[7].startswith("p"):
        return None
    return "_".join(parts[7:-1])  # e.g. "Schroeder", "GOLI_v2"


images = [f for f in all_files if algorithm_of(f) is None]
masks = [f for f in all_files if algorithm_of(f) is not None]
print("images:", len(images))
print("masks: ", len(masks))

mask_counts = {}
for f in masks:
    mask_counts[algorithm_of(f)] = mask_counts.get(algorithm_of(f), 0) + 1
print("masks per algorithm:", mask_counts)

for f in images[:3]:
    print(f.name)

In [ ]:
ALGORITHMS = ["Schroeder", "Murphy", "Kumar-Roy" if "Kumar-Roy" in mask_counts else "GOLI_v2"]
files_by_name = {f.name: f for f in all_files}


def find_mask(image_path, algorithm):
    """The mask file of an image, or None if there is none."""
    name = image_path.name.replace("_p", f"_{algorithm}_p")
    return files_by_name.get(name)


def read(path):
    with rasterio.open(path) as f:
        return f.read()

### Reading a file name

`LC08_L1TP_124046_20200810_20200810_01_RT_p00367.tif`

| Part | Meaning |
|---|---|
| `LC08` | Landsat 8 |
| `L1TP` | processing level |
| `124046` | location on Landsat's world grid: path 124, row 046 |
| `20200810` | date the picture was taken (10 Aug 2020) |
| `p00367` | patch number: the full picture was cut into small squares |

Everything before `_p00367` identifies one full satellite picture (a **scene**).
A scene is about 185 km wide and is cut into many 256 × 256 patches.

In [ ]:
scenes = {f.name.split("_p")[0] for f in images}
print("scenes:", len(scenes))
print("patches per scene, on average:", round(len(images) / len(scenes), 1))

A full scene is roughly 7,600 × 7,600 pixels, which would give about
(7600 / 256)² ≈ 880 patches. If the number above is much smaller, the dataset
does not keep every patch of every scene.

## 2. Open one image

We pick an image with a large fire to explore in the next sections.

In [ ]:
# Among the first 100 images with a mask from all three algorithms,
# take the one with the most Schroeder fire pixels, so there is something to see.
candidates = [f for f in images if all(find_mask(f, a) for a in ALGORITHMS)][:100]
image_path = max(candidates, key=lambda f: read(find_mask(f, "Schroeder")).sum())
print(image_path.name)

with rasterio.open(image_path) as f:
    image = f.read()

print("shape:", image.shape)  # (bands, height, width)
print("type: ", image.dtype)
print("min:  ", image.min(), " max:", image.max())

- **10 bands**: the satellite measures light in several wavelength ranges, and
  each one is stored as a separate grayscale layer.
- **256 × 256 pixels**: each pixel covers 30 × 30 m, so a patch is about 7.7 km wide.
- **uint16**: whole numbers from 0 to 65,535. These are the sensor's raw
  brightness readings, not calibrated physical units.

The 10 bands in each file, in order:

| Position | Band | What it measures |
|---|---|---|
| 1 | B1 | coastal blue |
| 2 | B2 | blue |
| 3 | B3 | green |
| 4 | B4 | red |
| 5 | B5 | near infrared (healthy plants are bright) |
| 6 | B6 | shortwave infrared 1 (SWIR1) |
| 7 | B7 | shortwave infrared 2 (SWIR2) |
| 8 | B9 | cirrus (thin high clouds) |
| 9 | B10 | thermal infrared 1 (heat) |
| 10 | B11 | thermal infrared 2 (heat) |

B8 (a sharper black-and-white band) is not included.

## 3. Look at each band

In [ ]:
BAND_NAMES = ["B1", "B2", "B3", "B4", "B5", "B6", "B7", "B9", "B10", "B11"]

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for i, ax in enumerate(axes.flat):
    ax.imshow(image[i], cmap="gray")
    ax.set_title(BAND_NAMES[i])
    ax.axis("off")
plt.show()

Things to notice: the visible bands (B1–B4) look alike; B9 is mostly dark
because it only sees high clouds; the thermal bands (B10, B11) look blurry
because the thermal sensor has coarser pixels (100 m) that were resized to 30 m.

## 4. Make color pictures

A screen shows three colors (red, green, blue), so we choose three bands and
display them as red, green and blue.

- **Natural color**: B4, B3, B2 (red, green, blue), close to what your eyes see.
- **False color**: B7, B6, B2. The paper's model uses only these three bands.
  Fire is very bright in B7 and B6 (shortwave infrared), so it shows up red or
  orange.

Raw values are spread unevenly, so each band is stretched to fill the 0–1
display range (darkest 2% → black, brightest 2% → white).

In [ ]:
def band(image, name):
    return image[BAND_NAMES.index(name)].astype(float)


def stretch(values):
    low, high = np.percentile(values, [2, 98])
    return np.clip((values - low) / (high - low + 1e-9), 0, 1)


def color_picture(image, red, green, blue):
    return np.dstack([stretch(band(image, red)), stretch(band(image, green)), stretch(band(image, blue))])


fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(color_picture(image, "B4", "B3", "B2"))
axes[0].set_title("natural color (B4, B3, B2)")
axes[1].imshow(color_picture(image, "B7", "B6", "B2"))
axes[1].set_title("false color (B7, B6, B2)")
for ax in axes:
    ax.axis("off")
plt.show()

What to look for:

- **Smoke** is white or gray in natural color but nearly disappears (or turns
  blue) in false color. Shortwave infrared passes through smoke, which is why
  these bands can see fires that the eye cannot.
- **Fire** is bright red, orange or yellow in false color. Very hot fire can max
  out the sensor (value 65,535, see the `max` printed in section 2) and look
  white.
- **Burned ground** (scars from earlier fires) looks dark red or purple. It is
  not active fire, and a good detector should ignore it.

## 5. Look at the fire masks of this image

`find_mask` (defined in section 1) inserts the algorithm name before the patch
number: `..._RT_p00367.tif` → `..._RT_Schroeder_p00367.tif`.

In [ ]:
picture = color_picture(image, "B7", "B6", "B2")
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(picture)
axes[0].set_title("false color")
for ax, algorithm in zip(axes[1:], ALGORITHMS):
    mask = read(find_mask(image_path, algorithm))[0]
    ax.imshow(mask, cmap="gray")
    ax.set_title(f"{algorithm}: {mask.sum()} fire pixels")
for ax in axes:
    ax.axis("off")
plt.show()

print("values in a mask:", np.unique(mask))

Fires are often only a few pixels, too small to see at this size. Let's zoom in
on the fire and paint Schroeder's fire pixels in cyan on top of the picture.

In [ ]:
mask = read(find_mask(image_path, "Schroeder"))[0]
rows, cols = np.nonzero(mask)
r, c = int(rows.mean()), int(cols.mean())  # center of the fire pixels
top = min(max(r - 25, 0), 256 - 50)  # keep the 50 x 50 window inside the image
left = min(max(c - 25, 0), 256 - 50)
zoom = picture[top:top + 50, left:left + 50]

painted = picture.copy()
painted[mask > 0] = [0, 1, 1]  # cyan
zoom_painted = painted[top:top + 50, left:left + 50]

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(zoom)
axes[0].set_title("zoom: false color")
axes[1].imshow(zoom_painted)
axes[1].set_title("zoom: mask in cyan")
for ax in axes:
    ax.axis("off")
plt.show()

The cyan pixels should cover the red/orange spots. To try another image, replace
the `image_path = max(...)` line in section 2 with `image_path = images[5]` (or
any number) and re-run sections 2 to 5, so the image and its masks match.

## 6. Why does fire show up in B7 and B6?

Compare the average value of each band on fire pixels and on non-fire pixels,
using 50 images and their Schroeder masks.

In [ ]:
fire_values, other_values = [], []
for path in images[:50]:
    mask_path = find_mask(path, "Schroeder")
    if mask_path is None:
        continue
    img = read(path)
    fire = read(mask_path)[0] > 0
    if fire.sum() == 0:
        continue
    fire_values.append(img[:, fire].mean(axis=1))
    other_values.append(img[:, ~fire].mean(axis=1))

fire_mean = np.mean(fire_values, axis=0)
other_mean = np.mean(other_values, axis=0)

x = np.arange(10)
plt.figure(figsize=(10, 4))
plt.bar(x - 0.2, other_mean, width=0.4, label="not fire")
plt.bar(x + 0.2, fire_mean, width=0.4, label="fire")
plt.xticks(x, BAND_NAMES)
plt.ylabel("average raw value")
plt.legend()
plt.show()

for name, f, o in zip(BAND_NAMES, fire_mean, other_mean):
    print(f"{name:>4}: fire is {f / o:.2f}x brighter")

Burning material is hot enough to emit light in the shortwave infrared, so B7
and B6 jump on fire pixels while the visible bands barely change. This is why
the paper picks B7, B6, B2 as the model's input, and it also confirms that the
bands in the file are in the order listed above.

## 7. How rare is fire?

Count, for each algorithm, how many mask files there are and what share of
pixels is fire. This can take a minute on the full regions.

In [ ]:
for algorithm in ALGORITHMS:
    files = [f for f in masks if algorithm_of(f) == algorithm]
    fire_pixels = [int(read(f).sum()) for f in files]
    total_pixels = len(images) * 256 * 256
    print(f"{algorithm:>10}: {len(files):>6} mask files"
          f" | fire pixels: {sum(fire_pixels) / total_pixels:.4%} of all pixels"
          f" | masks with only 1 fire pixel: {sum(n == 1 for n in fire_pixels)}")

Fire is a tiny fraction of all pixels. A model that always answers "no fire"
would be more than 99.9% accurate and completely useless. This is why the
paper reports precision, recall and F1 instead of accuracy, and why this
imbalance matters for the loss function later.

Also note how many images have a mask file at all. There is **no mask file for
patches where the algorithm found no fire**.

In [ ]:
for algorithm in ALGORITHMS:
    with_mask = sum(find_mask(f, algorithm) is not None for f in images)
    print(f"{algorithm:>10}: {with_mask} of {len(images)} images have a mask file")

## 8. Do the three algorithms agree?

For the image from section 2, count pixels marked as fire by all three
algorithms, by two, and by only one. The paper's "voting" mask keeps pixels with at least two votes.

In [ ]:
votes = np.zeros((256, 256), dtype=int)
for algorithm in ALGORITHMS:
    votes += read(find_mask(image_path, algorithm))[0] > 0

for n in [1, 2, 3]:
    print(f"pixels marked by exactly {n} algorithm(s): {(votes == n).sum()}")

plt.imshow(votes, cmap="viridis")
plt.colorbar(label="number of algorithms saying fire")
plt.axis("off")
plt.show()

## 9. The manually annotated images

A separate set of images where people drew the fires by hand. It is used only
to test the final models, never to train them. Manual masks have `_v1_` in the
name.

In [ ]:
manual_folder = DATA / "manual_annotations" if (DATA / "manual_annotations").exists() else DATA / "samples"
manual_masks = sorted(manual_folder.rglob("*_v1_*.tif"))
print(len(manual_masks), "manual masks")

if manual_masks:
    manual_mask_path = manual_masks[0]
    image_name = manual_mask_path.name.replace("_v1_", "_")
    manual_image_path = next(manual_folder.rglob(image_name))

    manual_image = read(manual_image_path)
    manual_mask = read(manual_mask_path)[0]

    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(color_picture(manual_image, "B7", "B6", "B2"))
    axes[0].set_title("false color")
    axes[1].imshow(manual_mask, cmap="gray")
    axes[1].set_title(f"drawn by hand: {manual_mask.sum()} fire pixels")
    for ax in axes:
        ax.axis("off")
    plt.show()

## 10. Several patches and their masks

Eight patches with a mask from all three algorithms, from the smallest fire to
the largest, zoomed in on the fire. In every row the cyan pixels should sit on
the bright orange or red spots; if they don't, the band order or the masks are
wrong. The figure is saved to `results/figures/patches_and_masks.png`.

In [ ]:
# 8 patches with a mask from all three algorithms, from the smallest fire to
# the largest (fire size = fire pixels summed over the three masks).
with_all_masks = [f for f in images if all(find_mask(f, a) for a in ALGORITHMS)][:300]
fire_size = {f: sum(int(read(find_mask(f, a)).sum()) for a in ALGORITHMS) for f in with_all_masks}
by_size = sorted(with_all_masks, key=lambda f: fire_size[f])
chosen = [by_size[i] for i in np.linspace(0, len(by_size) - 1, 8).astype(int)]

SIZE = 64  # zoom window in pixels (about 1.9 km)
fig, axes = plt.subplots(len(chosen), 4, figsize=(10, 2.7 * len(chosen)))
for row_axes, path in zip(axes, chosen):
    picture = color_picture(read(path), "B7", "B6", "B2")
    fire_masks = [read(find_mask(path, a))[0] > 0 for a in ALGORITHMS]

    # Center the zoom on the fire pixel with the most fire around it.
    votes = np.sum(fire_masks, axis=0)
    most_fire = -1
    for r, c in np.argwhere(votes > 0):
        top = min(max(r - SIZE // 2, 0), 256 - SIZE)  # keep the window inside the patch
        left = min(max(c - SIZE // 2, 0), 256 - SIZE)
        fire_around = votes[top:top + SIZE, left:left + SIZE].sum()
        if fire_around > most_fire:
            most_fire = fire_around
            window = (slice(top, top + SIZE), slice(left, left + SIZE))

    row_axes[0].imshow(picture[window])
    row_axes[0].set_title(f"location {path.name[10:16]}, patch {path.stem[-5:]}", fontsize=9)
    for ax, algorithm, fire in zip(row_axes[1:], ALGORITHMS, fire_masks):
        painted = picture.copy()
        painted[fire] = [0, 1, 1]  # cyan
        ax.imshow(painted[window])
        ax.set_title(f"{algorithm}: {fire.sum()} px", fontsize=9)
for ax in axes.flat:
    ax.axis("off")
fig.tight_layout()
Path("../results/figures").mkdir(parents=True, exist_ok=True)
fig.savefig("../results/figures/patches_and_masks.png", dpi=120)
plt.show()

## What I learned

Write your own notes here: number of images and scenes, how rare fire is, how
much the algorithms agree, anything surprising.